# Transformer Architectures -- Foundations

> **Section 01** -- Topic 04 -- `foundations` 
> **Prerequisites:** `03-attention-mechanisms` 
> **Objective:** Understand and implement the three major transformer variants (encoder-decoder, encoder-only, decoder-only), their core sub-layers, and how they compose into a working language model.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## 1. The Original Transformer

The 2017 paper *"Attention Is All You Need"* (Vaswani et al.) introduced the **encoder-decoder** transformer. The key insight: replace recurrence entirely with self-attention.

### Architecture overview

```
Input Tokens --> [Encoder x N] --> Encoder Output
                                        |
                                        v
Target Tokens --> [Decoder x N] ------> Output Probs
```

Each **encoder block** contains:
1. Multi-head self-attention
2. Feed-forward network
3. Layer normalization + residual connections around each

Each **decoder block** adds a **cross-attention** sub-layer between self-attention and FFN, attending to the encoder output.

### Why this matters

Understanding the original design clarifies *why* later models dropped the encoder (GPT) or the decoder (BERT). The encoder-decoder structure is still used in models like T5, BART, and many machine translation systems.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head attention used across all transformer variants."""
    
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None):
        B, T, C = query.shape
        
        # Project and reshape to (B, n_heads, T, d_k)
        Q = self.W_q(query).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Combine heads
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.W_o(context), attn_weights

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """Original Transformer encoder block (post-norm)."""
    
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, src_mask=None):
        # Self-attention + residual + norm (post-norm)
        attn_out, _ = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.dropout(attn_out))
        # FFN + residual + norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x

In [ ]:
class TransformerDecoderBlock(nn.Module):
    """Original Transformer decoder block with cross-attention."""
    
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # Masked self-attention
        attn_out, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_out))
        # Cross-attention over encoder output
        cross_out, _ = self.cross_attn(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(cross_out))
        # FFN
        ffn_out = self.ffn(x)
        x = self.norm3(x + ffn_out)
        return x

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding from the original paper."""
    
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class EncoderDecoder(nn.Module):
    """Complete encoder-decoder transformer."""
    
    def __init__(self, src_vocab, tgt_vocab, d_model=256, n_heads=8, n_layers=3, d_ff=1024, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.src_embed = nn.Embedding(src_vocab, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.output_proj = nn.Linear(d_model, tgt_vocab)
    
    def encode(self, src, src_mask=None):
        x = self.pos_enc(self.src_embed(src) * math.sqrt(self.d_model))
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        return x
    
    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        x = self.pos_enc(self.tgt_embed(tgt) * math.sqrt(self.d_model))
        for layer in self.decoder_layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.output_proj(x)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_out = self.encode(src, src_mask)
        return self.decode(tgt, enc_out, src_mask, tgt_mask)


# Quick test
model = EncoderDecoder(src_vocab=1000, tgt_vocab=1000)
src = torch.randint(0, 1000, (2, 10))
tgt = torch.randint(0, 1000, (2, 8))

# Create causal mask for decoder
tgt_len = tgt.size(1)
causal_mask = torch.tril(torch.ones(tgt_len, tgt_len)).unsqueeze(0).unsqueeze(0)
logits = model(src, tgt, tgt_mask=causal_mask)
print(f"Encoder-Decoder output shape: {logits.shape}  (batch=2, tgt_len=8, vocab=1000)")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 2. Encoder-Only: BERT-Style

BERT (Devlin et al., 2019) keeps only the encoder stack and trains with **Masked Language Modeling (MLM)**: randomly mask 15% of tokens, predict them from bidirectional context.

### Why encoder-only?

For tasks that require understanding the full context of a sentence (classification, NER, QA), bidirectional attention is more powerful than causal attention. The encoder sees all tokens simultaneously.

### MLM objective

```
Input:  The [MASK] sat on the [MASK]
Target: The  cat  sat on the  mat
Loss: only computed on masked positions
```

In [ ]:
class BERTEncoder(nn.Module):
    """Simplified BERT: encoder-only transformer with MLM head."""
    
    def __init__(self, vocab_size: int, d_model: int = 256, n_heads: int = 8,
                 n_layers: int = 4, d_ff: int = 1024, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)  # learned positions (BERT uses learned, not sinusoidal)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        # MLM head: project back to vocab
        self.mlm_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Linear(d_model, vocab_size),
        )
    
    def forward(self, input_ids, mask=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.token_embed(input_ids) + self.pos_embed(positions)
        
        for layer in self.layers:
            x = layer(x, mask)
        x = self.norm(x)
        return self.mlm_head(x)  # (B, T, vocab_size)


# Demonstrate MLM
VOCAB_SIZE = 5000
MASK_TOKEN = 4999

bert = BERTEncoder(vocab_size=VOCAB_SIZE)
input_ids = torch.randint(0, 4999, (4, 32))  # batch of 4, seq_len 32

# Mask 15% of tokens
mask_prob = 0.15
mask_positions = torch.rand_like(input_ids.float()) < mask_prob
labels = input_ids.clone()
labels[~mask_positions] = -100  # only compute loss on masked positions
input_ids[mask_positions] = MASK_TOKEN

logits = bert(input_ids)
loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), labels.view(-1), ignore_index=-100)

print(f"BERT output shape: {logits.shape}")
print(f"Masked tokens: {mask_positions.sum().item()} / {input_ids.numel()}")
print(f"MLM loss: {loss.item():.4f}")
print(f"Parameters: {sum(p.numel() for p in bert.parameters()):,}")

---
## 3. Decoder-Only: GPT-Style

GPT (Radford et al., 2018) uses only the decoder stack with **causal (autoregressive) attention**: each token can only attend to previous tokens. The training objective is next-token prediction.

### Why decoder-only won

Decoder-only models turned out to scale better for generation tasks. Key reasons:
- Simpler architecture (no cross-attention)
- Natural fit for generation -- predict one token at a time
- Unified pretraining objective (next-token prediction) works for everything
- With sufficient scale, in-context learning emerges

### Causal mask

```
Token 0: can attend to [0]
Token 1: can attend to [0, 1]
Token 2: can attend to [0, 1, 2]
...
```

In [ ]:
class GPTBlock(nn.Module):
    """GPT-style decoder block: causal self-attention + FFN (post-norm)."""
    
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, causal_mask):
        attn_out, _ = self.attn(x, x, x, causal_mask)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x


class GPT(nn.Module):
    """Simplified GPT: decoder-only transformer with LM head."""
    
    def __init__(self, vocab_size: int, d_model: int = 256, n_heads: int = 8,
                 n_layers: int = 4, d_ff: int = 1024, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([
            GPTBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        # Weight tying: share embedding and output weights
        self.lm_head.weight = self.token_embed.weight
    
    def forward(self, input_ids, targets=None):
        B, T = input_ids.shape
        assert T <= self.max_len, f"Sequence length {T} exceeds max_len {self.max_len}"
        
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.token_embed(input_ids) + self.pos_embed(positions)
        
        # Causal mask: lower triangular
        causal_mask = torch.tril(torch.ones(T, T, device=input_ids.device)).unsqueeze(0).unsqueeze(0)
        
        for layer in self.layers:
            x = layer(x, causal_mask)
        x = self.norm(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


# Test GPT
gpt = GPT(vocab_size=VOCAB_SIZE)
input_ids = torch.randint(0, VOCAB_SIZE, (4, 32))
targets = torch.randint(0, VOCAB_SIZE, (4, 32))
logits, loss = gpt(input_ids, targets)
print(f"GPT output shape: {logits.shape}")
print(f"LM loss: {loss.item():.4f} (random baseline ~ {math.log(VOCAB_SIZE):.4f})")
print(f"Parameters: {sum(p.numel() for p in gpt.parameters()):,}")

### Architecture comparison

| Property | Encoder-Decoder | Encoder-Only (BERT) | Decoder-Only (GPT) |
|----------|----------------|---------------------|--------------------|
| Attention | Bidirectional + causal + cross | Bidirectional | Causal |
| Pretraining | Seq2seq denoising | MLM | Next-token prediction |
| Best for | Translation, summarization | Classification, NER | Generation, few-shot |
| Examples | T5, BART, mBART | BERT, RoBERTa, DeBERTa | GPT, Llama, Claude |

---
## 4. Feed-Forward Networks

Every transformer block contains a **position-wise feed-forward network** (FFN). Despite being simple, the FFN holds roughly **2/3 of the parameters** in a transformer block.

### Standard FFN

$$\text{FFN}(x) = W_2 \cdot \text{act}(W_1 x + b_1) + b_2$$

where $W_1 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$ and $W_2 \in \mathbb{R}^{d_{\text{ff}} \times d_{\text{model}}}$.

The standard expansion ratio is $d_{\text{ff}} = 4 \times d_{\text{model}}$.

### Why 4x?

The original paper used 4x (512 -> 2048). This ratio was found empirically to give a good balance between expressivity and compute. Modern models sometimes use different ratios (e.g., Llama uses ~2.7x with SwiGLU, which has 3 weight matrices).

In [ ]:
class FFN_ReLU(nn.Module):
    """Original Transformer FFN with ReLU."""
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.w2(F.relu(self.w1(x)))


class FFN_GELU(nn.Module):
    """GPT-2 style FFN with GELU activation."""
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.w2(F.gelu(self.w1(x)))


# Compare activations
x = torch.linspace(-3, 3, 200)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ReLU vs GELU
axes[0].plot(x.numpy(), F.relu(x).numpy(), label='ReLU', linewidth=2)
axes[0].plot(x.numpy(), F.gelu(x).numpy(), label='GELU', linewidth=2)
axes[0].set_title('Activation Functions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].axvline(x=0, color='k', linewidth=0.5)

# Parameter breakdown
d_model = 256
d_ff = 1024  # 4x
n_heads = 8
attn_params = 4 * d_model * d_model  # Q, K, V, O
ffn_params = 2 * d_model * d_ff  # W1, W2 (ignoring biases for clarity)
total = attn_params + ffn_params

axes[1].bar(['Attention\n(Q,K,V,O)', 'FFN\n(W1,W2)'], [attn_params, ffn_params],
           color=['#2196F3', '#FF9800'])
axes[1].set_title(f'Parameter Distribution per Block (d_model={d_model})')
axes[1].set_ylabel('Parameters')
for i, v in enumerate([attn_params, ffn_params]):
    axes[1].text(i, v + 5000, f'{v:,}\n({v/total:.0%})', ha='center', fontsize=10)

plt.tight_layout()
plt.show()
print(f"FFN holds {ffn_params/total:.1%} of block parameters (excluding norms)")

In [ ]:
# Benchmark FFN variants
d_model, d_ff = 512, 2048
ffn_relu = FFN_ReLU(d_model, d_ff)
ffn_gelu = FFN_GELU(d_model, d_ff)

x = torch.randn(8, 128, d_model)

import time

def benchmark(module, x, name, n_iters=1000):
    # Warmup
    for _ in range(100):
        _ = module(x)
    start = time.perf_counter()
    for _ in range(n_iters):
        _ = module(x)
    elapsed = time.perf_counter() - start
    print(f"{name}: {elapsed/n_iters*1000:.3f} ms/iter")

benchmark(ffn_relu, x, "FFN-ReLU")
benchmark(ffn_gelu, x, "FFN-GELU")

---
## 5. Layer Normalization

Layer normalization is critical for training stability. It normalizes activations across the feature dimension (not batch dimension like BatchNorm).

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

### Pre-Norm vs Post-Norm

The original transformer used **post-norm**: normalize after the residual addition.
```
Post-norm: x = LayerNorm(x + Sublayer(x))
```

Modern models (GPT-2 onward) use **pre-norm**: normalize before the sublayer.
```
Pre-norm: x = x + Sublayer(LayerNorm(x))
```

### Why pre-norm is preferred

- More stable gradients at initialization -- the residual stream passes through cleanly
- Can train deeper models without warmup tricks
- Gradient magnitude is more uniform across layers

In [ ]:
class ManualLayerNorm(nn.Module):
    """Implement LayerNorm from scratch to understand the math."""
    
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps
    
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta


# Verify our implementation matches PyTorch
d_model = 256
x = torch.randn(4, 32, d_model)

our_ln = ManualLayerNorm(d_model)
torch_ln = nn.LayerNorm(d_model)

# Copy weights so we can compare
torch_ln.weight.data = our_ln.gamma.data.clone()
torch_ln.bias.data = our_ln.beta.data.clone()

our_out = our_ln(x)
torch_out = torch_ln(x)

print(f"Max difference: {(our_out - torch_out).abs().max().item():.2e}")
print("Implementation matches PyTorch!" if (our_out - torch_out).abs().max() < 1e-5 else "Mismatch!")

In [ ]:
class PreNormBlock(nn.Module):
    """Transformer block with pre-norm (GPT-2 style)."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        # Pre-norm: normalize BEFORE sublayer, residual is clean
        attn_out, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x), mask)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x


class PostNormBlock(nn.Module):
    """Transformer block with post-norm (original paper)."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        # Post-norm: normalize AFTER residual addition
        attn_out, _ = self.attn(x, x, x, mask)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ffn(x))
        return x


print("Pre-norm block:  x = x + Sublayer(LayerNorm(x))")
print("Post-norm block: x = LayerNorm(x + Sublayer(x))")

In [ ]:
# Visualize gradient flow through pre-norm vs post-norm stacks
def measure_gradient_norms(block_class, n_layers=12, d_model=128, n_heads=4, d_ff=512):
    """Stack N blocks and measure gradient norm at each layer."""
    blocks = nn.ModuleList([block_class(d_model, n_heads, d_ff) for _ in range(n_layers)])
    
    x = torch.randn(2, 16, d_model, requires_grad=True)
    T = x.size(1)
    mask = torch.tril(torch.ones(T, T)).unsqueeze(0).unsqueeze(0)
    
    # Forward
    activations = [x]
    h = x
    for block in blocks:
        h = block(h, mask)
        activations.append(h)
    
    # Backward from loss
    loss = h.sum()
    loss.backward()
    
    # Collect gradient norms of each block's attention output projection
    grad_norms = []
    for block in blocks:
        grad_norm = block.attn.W_o.weight.grad.norm().item()
        grad_norms.append(grad_norm)
    
    return grad_norms

pre_norms = measure_gradient_norms(PreNormBlock, n_layers=12)
post_norms = measure_gradient_norms(PostNormBlock, n_layers=12)

fig, ax = plt.subplots(figsize=(10, 5))
layers = range(1, 13)
ax.plot(layers, pre_norms, 'o-', label='Pre-Norm', linewidth=2, markersize=6)
ax.plot(layers, post_norms, 's-', label='Post-Norm', linewidth=2, markersize=6)
ax.set_xlabel('Layer (1 = closest to input)', fontsize=12)
ax.set_ylabel('Gradient Norm (W_o)', fontsize=12)
ax.set_title('Gradient Flow: Pre-Norm vs Post-Norm (12 layers)', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print(f"Pre-norm gradient ratio (layer 1 / layer 12): {pre_norms[0]/pre_norms[-1]:.2f}x")
print(f"Post-norm gradient ratio (layer 1 / layer 12): {post_norms[0]/post_norms[-1]:.2f}x")

---
## 6. Residual Connections

Residual connections (He et al., 2015) are the "highway" that lets gradients flow directly through the network. Without them, deep transformers are essentially untrainable.

$$\text{output} = x + \text{Sublayer}(x)$$

### Why they work

The residual connection creates a direct path for gradients during backpropagation:

$$\frac{\partial \text{output}}{\partial x} = I + \frac{\partial \text{Sublayer}(x)}{\partial x}$$

The identity term $I$ means gradients can always flow, even if the sublayer's gradients vanish. This transforms the optimization landscape from "learn a function" to "learn a residual correction."

In [ ]:
class BlockWithResidual(nn.Module):
    """Transformer block WITH residual connections."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        return x + self.ffn(self.norm(x))  # residual!


class BlockWithoutResidual(nn.Module):
    """Transformer block WITHOUT residual connections."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        return self.ffn(self.norm(x))  # no residual


def train_simple_model(block_class, n_layers=8, n_steps=200):
    """Train a stack of blocks on a simple regression task."""
    d_model, d_ff = 64, 256
    blocks = nn.Sequential(*[block_class(d_model, d_ff) for _ in range(n_layers)])
    head = nn.Linear(d_model, 1)
    model = nn.Sequential(blocks, head)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # Simple regression task
    X = torch.randn(64, d_model)
    y = X[:, 0:1] + X[:, 1:2] * 0.5  # simple target
    
    losses = []
    for step in range(n_steps):
        pred = model(X)
        loss = F.mse_loss(pred, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
    return losses

losses_with = train_simple_model(BlockWithResidual, n_layers=8)
losses_without = train_simple_model(BlockWithoutResidual, n_layers=8)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses_with, label='With Residual', linewidth=2)
ax.plot(losses_without, label='Without Residual', linewidth=2, linestyle='--')
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Training Stability: Residual vs No Residual (8 layers)', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print(f"Final loss WITH residual: {losses_with[-1]:.6f}")
print(f"Final loss WITHOUT residual: {losses_without[-1]:.6f}")

In [ ]:
# Visualize the residual stream concept
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Diagram: signal magnitude through layers
d_model = 128
n_layers = 12

# With residual: measure activation norms
x = torch.randn(1, 1, d_model)
norms_with = [x.norm().item()]
blocks_res = nn.ModuleList([BlockWithResidual(d_model, d_model * 4) for _ in range(n_layers)])
h = x
for block in blocks_res:
    h = block(h)
    norms_with.append(h.norm().item())

# Without residual
norms_without = [x.norm().item()]
blocks_nores = nn.ModuleList([BlockWithoutResidual(d_model, d_model * 4) for _ in range(n_layers)])
h = x
for block in blocks_nores:
    h = block(h)
    norms_without.append(h.norm().item())

axes[0].plot(range(n_layers + 1), norms_with, 'o-', label='With Residual', linewidth=2)
axes[0].plot(range(n_layers + 1), norms_without, 's-', label='Without Residual', linewidth=2)
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('Activation Norm')
axes[0].set_title('Activation Magnitudes Through Layers')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Diagram: gradient norms backward
x_res = torch.randn(1, 1, d_model, requires_grad=True)
h = x_res
intermediates_res = [h]
for block in blocks_res:
    h = block(h)
    intermediates_res.append(h)
h.sum().backward()
grad_norm_input_res = x_res.grad.norm().item()

x_nores = torch.randn(1, 1, d_model, requires_grad=True)
h = x_nores
for block in blocks_nores:
    h = block(h)
h.sum().backward()
grad_norm_input_nores = x_nores.grad.norm().item()

axes[1].bar(['With Residual', 'Without Residual'],
           [grad_norm_input_res, grad_norm_input_nores],
           color=['#4CAF50', '#F44336'])
axes[1].set_ylabel('Gradient Norm at Input')
axes[1].set_title('Gradient Reaching Input Layer (12 layers deep)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## 7. Putting It All Together

Let us now assemble a complete, modern decoder-only transformer (GPT-2 style with pre-norm), count its parameters carefully, and generate text from it.

### The full recipe

```
Token Embedding + Position Embedding
         |
    [Block x N]
         |--- LayerNorm -> Multi-Head Attention -> + (residual)
         |--- LayerNorm -> FFN(GELU) -> + (residual)
         |
    Final LayerNorm
         |
    LM Head (tied weights)
```

In [ ]:
class TransformerBlock(nn.Module):
    """Complete transformer block: pre-norm, GELU FFN, residual connections."""
    
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
    
    def forward(self, x, mask=None):
        # Pre-norm attention with residual
        normed = self.ln1(x)
        attn_out, _ = self.attn(normed, normed, normed, mask)
        x = x + attn_out
        # Pre-norm FFN with residual
        x = x + self.ffn(self.ln2(x))
        return x


class CompleteGPT(nn.Module):
    """Complete GPT-2 style transformer with all the pieces."""
    
    def __init__(self, vocab_size: int, d_model: int = 256, n_heads: int = 8,
                 n_layers: int = 6, d_ff: int = 1024, max_len: int = 256, dropout: float = 0.1):
        super().__init__()
        self.max_len = max_len
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying
        self.lm_head.weight = self.token_embed.weight
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, input_ids, targets=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.drop(self.token_embed(input_ids) + self.pos_embed(positions))
        
        causal_mask = torch.tril(torch.ones(T, T, device=input_ids.device)).unsqueeze(0).unsqueeze(0)
        
        for block in self.blocks:
            x = block(x, causal_mask)
        
        x = self.final_norm(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
    
    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0, top_k=None):
        """Autoregressive generation with optional top-k sampling."""
        for _ in range(max_new_tokens):
            # Crop to max_len if needed
            idx_cond = input_ids[:, -self.max_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
        return input_ids

In [ ]:
# Instantiate and count parameters
VOCAB_SIZE = 10000
model = CompleteGPT(
    vocab_size=VOCAB_SIZE,
    d_model=256,
    n_heads=8,
    n_layers=6,
    d_ff=1024,
    max_len=256,
)

# Detailed parameter count
def count_parameters(model):
    table = []
    total = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            count = param.numel()
            total += count
            table.append((name, list(param.shape), count))
    return table, total

table, total = count_parameters(model)

# Group by component
groups = {}
for name, shape, count in table:
    if 'token_embed' in name or 'pos_embed' in name:
        group = 'Embeddings'
    elif 'attn' in name:
        group = 'Attention'
    elif 'ffn' in name:
        group = 'FFN'
    elif 'ln' in name or 'norm' in name:
        group = 'LayerNorm'
    else:
        group = 'Other'
    groups[group] = groups.get(group, 0) + count

print(f"{'Component':<15} {'Parameters':>12} {'Fraction':>10}")
print("-" * 40)
for group, count in sorted(groups.items(), key=lambda x: -x[1]):
    print(f"{group:<15} {count:>12,} {count/total:>10.1%}")
print("-" * 40)
print(f"{'TOTAL':<15} {total:>12,}")
print(f"\nNote: LM head shares weights with token embedding (weight tying)")

In [ ]:
# Visualize parameter distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart of parameter groups
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336']
labels = list(groups.keys())
sizes = list(groups.values())
axes[0].pie(sizes, labels=labels, colors=colors[:len(labels)], autopct='%1.1f%%',
           startangle=90, textprops={'fontsize': 11})
axes[0].set_title(f'Parameter Distribution ({total:,} total)', fontsize=13)

# Per-layer breakdown
layer_attn = []
layer_ffn = []
layer_norm = []
for i in range(6):
    attn_p = sum(p.numel() for n, p in model.named_parameters() if f'blocks.{i}' in n and 'attn' in n)
    ffn_p = sum(p.numel() for n, p in model.named_parameters() if f'blocks.{i}' in n and 'ffn' in n)
    norm_p = sum(p.numel() for n, p in model.named_parameters() if f'blocks.{i}' in n and 'ln' in n)
    layer_attn.append(attn_p)
    layer_ffn.append(ffn_p)
    layer_norm.append(norm_p)

x_pos = range(6)
axes[1].bar(x_pos, layer_attn, label='Attention', color='#2196F3')
axes[1].bar(x_pos, layer_ffn, bottom=layer_attn, label='FFN', color='#FF9800')
axes[1].bar(x_pos, layer_norm, bottom=[a+f for a,f in zip(layer_attn, layer_ffn)],
           label='LayerNorm', color='#4CAF50')
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Parameters')
axes[1].set_title('Parameters per Layer')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Quick training on a synthetic task: memorize a pattern
# Generate repeating pattern data
pattern = torch.tensor([1, 2, 3, 4, 5, 6, 7, 8, 9, 10] * 10)  # repeating pattern

def get_batch(pattern, batch_size=16, seq_len=32):
    """Get random chunks from the pattern."""
    ix = torch.randint(0, len(pattern) - seq_len - 1, (batch_size,))
    x = torch.stack([pattern[i:i+seq_len] for i in ix])
    y = torch.stack([pattern[i+1:i+seq_len+1] for i in ix])
    return x, y

# Small model for fast training
small_model = CompleteGPT(
    vocab_size=20, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_len=64
)
optimizer = torch.optim.AdamW(small_model.parameters(), lr=3e-4)

losses = []
for step in range(300):
    x, y = get_batch(pattern)
    logits, loss = small_model(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 50 == 0:
        print(f"Step {step:3d}: loss = {loss.item():.4f}")

print(f"\nFinal loss: {losses[-1]:.4f}")

In [ ]:
# Generate from the trained model
small_model.eval()
prompt = torch.tensor([[1, 2, 3]])
generated = small_model.generate(prompt, max_new_tokens=20, temperature=0.5)
print(f"Prompt:    {prompt[0].tolist()}")
print(f"Generated: {generated[0].tolist()}")
print(f"Expected:  {[1,2,3,4,5,6,7,8,9,10,1,2,3,4,5,6,7,8,9,10,1,2,3]}")

# Plot training loss
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, linewidth=1.5, alpha=0.7)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Training a Small GPT on a Repeating Pattern')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary

In this notebook we built all three transformer variants from scratch:

| Concept | Key Takeaway |
|---------|-------------|
| **Encoder-Decoder** | Original design for seq2seq. Cross-attention bridges encoder and decoder. |
| **Encoder-Only (BERT)** | Bidirectional attention + MLM. Best for understanding tasks. |
| **Decoder-Only (GPT)** | Causal attention + next-token prediction. Scales best for generation. |
| **FFN** | Holds ~2/3 of block parameters. GELU preferred over ReLU. |
| **Layer Normalization** | Pre-norm (modern) gives more stable gradients than post-norm (original). |
| **Residual Connections** | Essential for gradient flow in deep networks. |
| **Complete GPT** | Token embed + position embed + N blocks + final norm + LM head. Weight tying saves parameters. |

### Next: `advanced.ipynb`

We will replace each classical component with its modern counterpart: RoPE instead of learned positions, RMSNorm instead of LayerNorm, SwiGLU instead of GELU FFN, and more.